#### 1 PSO Variant: PSO-LDIW

In [1]:
import numpy as np
import pyswarms as ps

# 1. 10-D Rastrigin function (minimization)
def rastrigin(X):
    # X: (n_particles, dimensions)
    A = 10
    return A * X.shape[1] + np.sum(X**2 - A * np.cos(2 * np.pi * X), axis=1)

# 2. 50-D One-Max function (maximization, but we minimize negative sum)
def onemax(X):
    # X: (n_particles, dimensions), values in [0,1]
    return -np.sum(X, axis=1)

# PSO-LDIW: inertia decreases linearly from 0.9 to 0.4
def run_pso_ldiw(func, dim, bounds, iters=100, n_particles=30, maximize=False):
    w_start, w_end = 0.9, 0.4
    c1, c2 = 2.0, 2.0

    # Custom options for PSO
    options = {'c1': c1, 'c2': c2, 'w': w_start}

    # Create optimizer
    optimizer = ps.single.GlobalBestPSO(
        n_particles=n_particles, dimensions=dim, options=options, bounds=bounds
    )

    # Store best cost at each iteration
    best_costs = []
    best_positions = []

    # Custom optimization loop to update inertia weight
    for i in range(iters):
        # Linearly decrease inertia
        w = w_start - (w_start - w_end) * (i / (iters - 1))
        optimizer.options['w'] = w

        # Perform one step of optimization
        cost, pos = optimizer.optimize(func, iters=1, verbose=False)
        best_costs.append(cost)
        best_positions.append(pos)

    # The last returned cost and pos from optimizer.optimize are the global best
    final_cost = best_costs[-1]
    final_pos = best_positions[-1]

    if maximize:
        # For maximization, return negative of best cost
        return -final_cost, final_pos, best_costs
    else:
        return final_cost, final_pos, best_costs

# Run 10-D Rastrigin
np.random.seed(42)
rastrigin_bounds = (np.full(10, -5), np.full(10, 5))
rastrigin_best_cost, rastrigin_best_pos, rastrigin_history = run_pso_ldiw(
    rastrigin, dim=10, bounds=rastrigin_bounds, iters=100, n_particles=30
)
print("10-D Rastrigin best cost:", rastrigin_best_cost)

# Run 50-D One-Max
onemax_bounds = (np.zeros(50), np.ones(50))
onemax_best_score, onemax_best_pos, onemax_history = run_pso_ldiw(
    onemax, dim=50, bounds=onemax_bounds, iters=100, n_particles=30, maximize=True
)
print("50-D One-Max best score:", onemax_best_score)


10-D Rastrigin best cost: 8.013315951646817
50-D One-Max best score: 31.185402432647006


#### 2 PSO Variant: TVAC-PSO

In [2]:
def run_pso_tvac(func, dim, bounds, iters=100, n_particles=30, maximize=False):
    """
    PSO with Time Varying Acceleration Coefficients (TVAC-PSO).
    - c1 decreases from 3.5 to 0.5 (cognitive)
    - c2 increases from 0.5 to 3.5 (social)
    - inertia w decreases from 0.9 to 0.4
    """
    w_start, w_end = 0.9, 0.4
    c1_start, c1_end = 3.5, 0.5
    c2_start, c2_end = 0.5, 3.5

    # Initial options
    options = {'c1': c1_start, 'c2': c2_start, 'w': w_start}

    optimizer = ps.single.GlobalBestPSO(
        n_particles=n_particles, dimensions=dim, options=options, bounds=bounds
    )

    best_costs = []
    best_positions = []

    for i in range(iters):
        # Linearly vary coefficients
        w = w_start - (w_start - w_end) * (i / (iters - 1))
        c1 = c1_start - (c1_start - c1_end) * (i / (iters - 1))
        c2 = c2_start + (c2_end - c2_start) * (i / (iters - 1))
        optimizer.options['w'] = w
        optimizer.options['c1'] = c1
        optimizer.options['c2'] = c2

        cost, pos = optimizer.optimize(func, iters=1, verbose=False)
        best_costs.append(cost)
        best_positions.append(pos)

    final_cost = best_costs[-1]
    final_pos = best_positions[-1]

    if maximize:
        return -final_cost, final_pos, best_costs
    else:
        return final_cost, final_pos, best_costs

# Run 10-D Rastrigin with TVAC-PSO
np.random.seed(42)
rastrigin_tvac_best_cost, rastrigin_tvac_best_pos, rastrigin_tvac_history = run_pso_tvac(
    rastrigin, dim=10, bounds=rastrigin_bounds, iters=100, n_particles=30
)
print("10-D Rastrigin (TVAC-PSO) best cost:", rastrigin_tvac_best_cost)

# Run 50-D One-Max with TVAC-PSO
onemax_tvac_best_score, onemax_tvac_best_pos, onemax_tvac_history = run_pso_tvac(
    onemax, dim=50, bounds=onemax_bounds, iters=100, n_particles=30, maximize=True
)
print("50-D One-Max (TVAC-PSO) best score:", onemax_tvac_best_score)


10-D Rastrigin (TVAC-PSO) best cost: 11.901161818461262
50-D One-Max (TVAC-PSO) best score: 31.966832894974804


#### 3 Hyperparameter Tuning (Enumeration) - Inertia Weights w

In [3]:
import numpy as np

def has_converged(history, tol=1e-2, patience=10):
    """
    Check if the optimization has converged.
    Convergence is defined as the best cost not improving by more than tol
    over the *last* 'patience' iterations.
    Returns (converged: bool, convergence_iteration: int)
    If converged, convergence_iteration is the first iteration where the
    last 'patience' values are within tol of each other.
    If not converged, returns (False, len(history))
    """
    if len(history) < patience:
        return False, len(history)
    # Only check the last 'patience' values
    window = history[-patience:]
    if np.max(window) - np.min(window) <= tol:
        # Find the first iteration where this window started
        for i in range(len(history) - patience + 1):
            w = history[i:i+patience]
            if np.max(w) - np.min(w) <= tol:
                return True, i + patience
    return False, len(history)

w_values = np.arange(0.1, 1.0, 0.1)
n_runs = 20
iters = 200
n_particles = 30
c1 = c2 = 2.05

rastrigin_results = []

for w in w_values:
    best_costs = []
    convergence_iters = []
    for run in range(n_runs):
        np.random.seed(run)  # for reproducibility
        options = {'c1': c1, 'c2': c2, 'w': w}
        # Initialize particles randomly within bounds
        lb, ub = np.array(rastrigin_bounds[0]), np.array(rastrigin_bounds[1])
        init_pos = np.random.uniform(low=lb, high=ub, size=(n_particles, 10))
        optimizer = ps.single.GlobalBestPSO(
            n_particles=n_particles, dimensions=10, options=options, bounds=rastrigin_bounds, init_pos=init_pos
        )
        cost, pos = optimizer.optimize(rastrigin, iters=iters, verbose=False)
        # optimizer.cost_history is the best cost at each iteration
        history = optimizer.cost_history
        best_costs.append(cost)
        converged, conv_iter = has_converged(history)
        convergence_iters.append(conv_iter)
        print(f"Run {run+1} - w={w:.1f} - Best Cost: {cost:.4f} - Convergence Iteration: {conv_iter}")
    avg_best_cost = np.mean(best_costs)
    avg_conv_iter = np.mean(convergence_iters)
    rastrigin_results.append((w, avg_best_cost, avg_conv_iter))

print("Inertia Weight Sweep Results (10-D Rastrigin, c1=c2=2.05):")
print(" w    | Avg Best Cost | Avg Convergence Iteration")
print("-----------------------------------------------")
for w, avg_cost, avg_iter in rastrigin_results:
    print(f"{w:.1f}  | {avg_cost:.4f}     | {avg_iter:.1f}")


Run 1 - w=0.1 - Best Cost: 13.9325 - Convergence Iteration: 81
Run 2 - w=0.1 - Best Cost: 5.9703 - Convergence Iteration: 111
Run 3 - w=0.1 - Best Cost: 2.9856 - Convergence Iteration: 181
Run 4 - w=0.1 - Best Cost: 3.0078 - Convergence Iteration: 200
Run 5 - w=0.1 - Best Cost: 10.9445 - Convergence Iteration: 59
Run 6 - w=0.1 - Best Cost: 6.3993 - Convergence Iteration: 200
Run 7 - w=0.1 - Best Cost: 5.0008 - Convergence Iteration: 174
Run 8 - w=0.1 - Best Cost: 11.9909 - Convergence Iteration: 200
Run 9 - w=0.1 - Best Cost: 6.9653 - Convergence Iteration: 171
Run 10 - w=0.1 - Best Cost: 9.1146 - Convergence Iteration: 200
Run 11 - w=0.1 - Best Cost: 6.9649 - Convergence Iteration: 192
Run 12 - w=0.1 - Best Cost: 12.9345 - Convergence Iteration: 153
Run 13 - w=0.1 - Best Cost: 11.9839 - Convergence Iteration: 200
Run 14 - w=0.1 - Best Cost: 5.9698 - Convergence Iteration: 105
Run 15 - w=0.1 - Best Cost: 2.2602 - Convergence Iteration: 200
Run 16 - w=0.1 - Best Cost: 7.9636 - Convergen

#### 4 Hyperparameter Tuning (Enumeration) - Cognitive Factor c1

In [4]:
c1_values = np.arange(1.0, 3.01, 0.2)
w = 0.7
c2 = 2.05

c1_results = []

for c1 in c1_values:
    best_costs = []
    convergence_iters = []
    for run in range(n_runs):
        np.random.seed(run)  # for reproducibility
        options = {'c1': c1, 'c2': c2, 'w': w}
        # Initialize particles randomly within bounds
        lb, ub = np.array(rastrigin_bounds[0]), np.array(rastrigin_bounds[1])
        init_pos = np.random.uniform(low=lb, high=ub, size=(n_particles, 10))
        optimizer = ps.single.GlobalBestPSO(
            n_particles=n_particles, dimensions=10, options=options, bounds=rastrigin_bounds, init_pos=init_pos
        )
        cost, pos = optimizer.optimize(rastrigin, iters=iters, verbose=False)
        history = optimizer.cost_history
        best_costs.append(cost)
        converged, conv_iter = has_converged(history)
        convergence_iters.append(conv_iter)
        print(f"Run {run+1} - c1={c1:.2f} - Best Cost: {cost:.4f} - Convergence Iteration: {conv_iter}")
    avg_best_cost = np.mean(best_costs)
    avg_conv_iter = np.mean(convergence_iters)
    c1_results.append((c1, avg_best_cost, avg_conv_iter))

print("Cognitive Factor Sweep Results (10-D Rastrigin, w=0.9, c2=2.05):")
print(" c1   | Avg Best Cost | Avg Convergence Iteration")
print("-----------------------------------------------")
for c1, avg_cost, avg_iter in c1_results:
    print(f"{c1:.2f} | {avg_cost:.4f}     | {avg_iter:.1f}")


Run 1 - c1=1.00 - Best Cost: 4.0868 - Convergence Iteration: 200
Run 2 - c1=1.00 - Best Cost: 9.1779 - Convergence Iteration: 200
Run 3 - c1=1.00 - Best Cost: 8.1049 - Convergence Iteration: 200
Run 4 - c1=1.00 - Best Cost: 11.2277 - Convergence Iteration: 200
Run 5 - c1=1.00 - Best Cost: 4.3248 - Convergence Iteration: 200
Run 6 - c1=1.00 - Best Cost: 2.0804 - Convergence Iteration: 200
Run 7 - c1=1.00 - Best Cost: 9.0468 - Convergence Iteration: 200
Run 8 - c1=1.00 - Best Cost: 10.0094 - Convergence Iteration: 200
Run 9 - c1=1.00 - Best Cost: 4.4222 - Convergence Iteration: 200
Run 10 - c1=1.00 - Best Cost: 10.9446 - Convergence Iteration: 20
Run 11 - c1=1.00 - Best Cost: 12.6351 - Convergence Iteration: 200
Run 12 - c1=1.00 - Best Cost: 9.9665 - Convergence Iteration: 200
Run 13 - c1=1.00 - Best Cost: 11.9879 - Convergence Iteration: 200
Run 14 - c1=1.00 - Best Cost: 7.6129 - Convergence Iteration: 200
Run 15 - c1=1.00 - Best Cost: 8.9557 - Convergence Iteration: 24
Run 16 - c1=1.00

#### 5 Hyperparameter Tuning (Enumeration) - Global Factor c2

In [5]:
c2_values = np.arange(1.0, 3.01, 0.2)
w = 0.7
c1 = 2.05

c2_results = []

for c2 in c2_values:
    best_costs = []
    convergence_iters = []
    for run in range(n_runs):
        np.random.seed(run)  # for reproducibility
        options = {'c1': c1, 'c2': c2, 'w': w}
        # Initialize particles randomly within bounds
        lb, ub = np.array(rastrigin_bounds[0]), np.array(rastrigin_bounds[1])
        init_pos = np.random.uniform(low=lb, high=ub, size=(n_particles, 10))
        optimizer = ps.single.GlobalBestPSO(
            n_particles=n_particles, dimensions=10, options=options, bounds=rastrigin_bounds, init_pos=init_pos
        )
        cost, pos = optimizer.optimize(rastrigin, iters=iters, verbose=False)
        history = optimizer.cost_history
        best_costs.append(cost)
        converged, conv_iter = has_converged(history)
        convergence_iters.append(conv_iter)
        print(f"Run {run+1} - c2={c2:.2f} - Best Cost: {cost:.4f} - Convergence Iteration: {conv_iter}")
    avg_best_cost = np.mean(best_costs)
    avg_conv_iter = np.mean(convergence_iters)
    c2_results.append((c2, avg_best_cost, avg_conv_iter))

print("Global Factor Sweep Results (10-D Rastrigin, w=0.9, c1=2.05):")
print(" c2   | Avg Best Cost | Avg Convergence Iteration")
print("-----------------------------------------------")
for c2, avg_cost, avg_iter in c2_results:
    print(f"{c2:.2f} | {avg_cost:.4f}     | {avg_iter:.1f}")


Run 1 - c2=1.00 - Best Cost: 7.0591 - Convergence Iteration: 200
Run 2 - c2=1.00 - Best Cost: 2.1340 - Convergence Iteration: 200
Run 3 - c2=1.00 - Best Cost: 4.2587 - Convergence Iteration: 200
Run 4 - c2=1.00 - Best Cost: 7.4433 - Convergence Iteration: 200
Run 5 - c2=1.00 - Best Cost: 3.0583 - Convergence Iteration: 200
Run 6 - c2=1.00 - Best Cost: 9.9676 - Convergence Iteration: 200
Run 7 - c2=1.00 - Best Cost: 5.1407 - Convergence Iteration: 200
Run 8 - c2=1.00 - Best Cost: 5.1543 - Convergence Iteration: 200
Run 9 - c2=1.00 - Best Cost: 6.2797 - Convergence Iteration: 200
Run 10 - c2=1.00 - Best Cost: 7.9699 - Convergence Iteration: 24
Run 11 - c2=1.00 - Best Cost: 9.0139 - Convergence Iteration: 200
Run 12 - c2=1.00 - Best Cost: 5.0004 - Convergence Iteration: 200
Run 13 - c2=1.00 - Best Cost: 10.1613 - Convergence Iteration: 200
Run 14 - c2=1.00 - Best Cost: 6.9873 - Convergence Iteration: 200
Run 15 - c2=1.00 - Best Cost: 1.1185 - Convergence Iteration: 200
Run 16 - c2=1.00 - 